In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from part2 import RaceSimulator, plot_tires, find_best_strategy

sim = RaceSimulator()
sim.tires

Load the Part 1 Excel file: tire table in `A3:E6`, strategy inputs in `A10:B15`, lap-by-lap simulation in `H3:R63`, total in `A18`.

In [ ]:
XLSX = 'Assignment_3a.xlsx'

excel_tires = pd.read_excel(XLSX, header=2, usecols='A:E', nrows=3, index_col=0)
excel_race = pd.read_excel(XLSX, header=2, usecols='H:R', nrows=60)
inputs = pd.read_excel(XLSX, header=None, usecols='A:B', skiprows=9, nrows=6)
excel_total = pd.read_excel(XLSX, header=None, usecols='A', skiprows=17, nrows=1).iloc[0, 0]

print(excel_tires, '\n')
print(inputs.dropna(), '\n')
print('Excel total race time added (A18):', excel_total)

The Excel strategy has pit stops on laps 15 and 23 with Soft → Medium → Hard, so the stint lengths are `(14, 8, 38)`.

## 2. Tire data and degradation plot

In [ ]:
print('Tire data matches Excel:',
      np.allclose(sim.tires.loc[excel_tires.index].to_numpy(dtype=float),
                  excel_tires.to_numpy(dtype=float)))
plot_tires(sim)

## Normal and special cases
Expected values come from Excel (case 1) or from an independent loop calculation plus 50 s for the two pit stops (other cases).

In [ ]:
def expected_time(stints, tires):
    """Independent check: plain loops + 2 pit stops of 25 s."""
    total = 50.0
    for n, t in zip(stints, tires):
        r = sim.tires.loc[t]
        for age in range(n):
            total += r.Tmin + (r.Tmax - r.Tmin) / (1 + np.exp(-r.alpha * (age - r.beta)))
    return total

tests = [
    ('Normal: Excel strategy', (14, 8, 38), ('Soft', 'Medium', 'Hard'), excel_total),
    ('Normal: equal stints',   (20, 20, 20), ('Medium', 'Hard', 'Medium'), None),
    ('Normal: mixed',          (10, 25, 25), ('Soft', 'Hard', 'Hard'), None),
    ('Special: 1-lap stints',  (1, 1, 58),   ('Soft', 'Medium', 'Hard'), None),
    ('Special: long 1st stint', (58, 1, 1),  ('Hard', 'Soft', 'Medium'), None),
    ('Special: repeated tire', (20, 20, 20), ('Soft', 'Soft', 'Hard'), None),
]
rows = []
for name, stints, tires, expected in tests:
    expected = expected if expected is not None else expected_time(stints, tires)
    actual = sim.simulate(stints, tires)
    rows.append({'case': name, 'stints': stints, 'tires': tires,
                 'expected': expected, 'actual': actual, 'pass': np.isclose(expected, actual)})
pd.DataFrame(rows)